In [1]:
!pip install rasterio
!pip install pyextremes


In [2]:
import zipfile
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
from pathlib import Path
from pyextremes import EVA
from pyextremes import plot_parameter_stability
import pyextremes
import warnings
import os

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

In [3]:
# Directory containing the CSV files
input_dir = '/Users/jackywong1105/Desktop/Project 1/WindReturnMap/downloads'
output_dir = '/Users/jackywong1105/Desktop/Project 1/WindReturnMap/results'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

In [7]:
return_periods = [10,25,50,100,150,250,500]
# return_periods = list(range(1, 1001))


In [8]:
# Prepare a dictionary to collect results by return period
results_by_return_period = {rp: [] for rp in return_periods}

# Initialize dictionary to store results for all grid points
# results = {}
all_summaries = []


below one is for return_periods = list(range(1, 1001))


In [6]:
# Process each CSV file
for filename in os.listdir(input_dir):
    # Check if file is a CSV and follows the grid_{long}_{lat}.csv pattern
    if filename.endswith('.csv') and filename.startswith('grid_'):
        filepath = os.path.join(input_dir, filename)

        # Extract longitude and latitude from filename
        lon_lat = filename.replace('.csv', '')

        # Read and preprocess data
        data = (
            pd
            .read_csv(filepath, index_col=2, parse_dates=True)
            .sort_index(ascending=True)
            .dropna()
        )

        data = data['max_wind_speed'].astype(float).squeeze()

        # Initialize Extremes object with the time series
        model = EVA(data=data)

        threshold_data = float(np.percentile(data, 98))

        # Extract peaks over threshold at 98% quantile
        model.get_extremes(
            method='POT',
            extremes_type='high',
            threshold=threshold_data,
            r="24h",
        )

        min_extremes_required = 2
        # Check if enough extremes for fitting
        if len(model.extremes) <= min_extremes_required:
            print(f"Not enough extremes extracted for {lon_lat}, skipping model fitting.")
            continue

        # Fit the GPD model to extracted extremes
        model.fit_model()

        # Get summary with return values and confidence intervals
        try:
            summary_df = model.get_summary(
                return_period=return_periods,
                return_period_size='365.2425D',
                alpha=0.95
            )

            # Reset index to make return_period a column
            summary_df = summary_df.reset_index().rename(columns={'index': 'return period'})
            # Add longitude_latitude column
            summary_df['longitude_latitude'] = lon_lat

            # Append to list of summaries
            all_summaries.append(summary_df)

        except Exception as e:
            print(f"Failed to get summary for {lon_lat}: {e}")
            continue

# Combine and pivot all summaries
if all_summaries:
    combined_df = pd.concat(all_summaries, ignore_index=True)

    # Pivot to get return periods as columns
    pivoted_df = combined_df.pivot(
        index='longitude_latitude',
        columns='return period',
        values='return value'
    ).reset_index()

    # Rename columns to match desired format
    pivoted_df.columns = [
        'longitude_latitude' if col == 'longitude_latitude'
        else f'return_level_{int(col)}years' for col in pivoted_df.columns
    ]

    # Save to a single CSV
    summary_output_path = os.path.join(output_dir, 'all_grid_summaries.csv')
    pivoted_df.to_csv(summary_output_path, index=False)

    print(f"Saved combined summary to {summary_output_path}")
else:
    print("No summaries were generated.")

Process SpawnPoolWorker-5:
Process SpawnPoolWorker-6:


KeyboardInterrupt: 

below one is for return_periods = [10,25,50,100,250,500]



In [9]:
# Process each CSV file
for filename in os.listdir(input_dir):
    if filename.endswith('.csv') and filename.startswith('grid_'):        
        filepath = os.path.join(input_dir, filename)

        # Extract longitude and latitude from filename
        lon_lat = filename.replace('.csv', '')

        data = (
                pd
                .read_csv(filepath, index_col=2, parse_dates=True)
                .sort_index(ascending=True)
                .dropna()
                )

        data = data['max_wind_speed'].astype(float).squeeze()

        # Initialize Extremes object with the time series
        model = EVA(data = data)

        threshold_data = float(np.percentile(data,98))

        # Extract peaks over threshold at 98% quantile
        model.get_extremes(
            method='POT',
            extremes_type='high',
            threshold=threshold_data,
            r="24h",
        )


        min_extremes_required = 2
        # Check if enough extremes for fitting
        if len(model.extremes) <= min_extremes_required:
          print(f"Not enough extremes extracted for {lon_lat}, skipping model fitting.")
          continue


        # Fit the GPD model to extracted extremes
        model.fit_model()

        for rp in return_periods:
            try:
                ret_level_tuple = model.get_return_value(return_period=rp)
                # Extract only the first value (return level) from the tuple
                ret_level = ret_level_tuple[0]
                results_by_return_period[rp].append({
                    'longitude_latitude': lon_lat,
                    'return_level': ret_level
                })
            except Exception as e:
                print(f"Failed for {filename} at {rp} years: {e}")



        # Get summary with return values and confidence intervals
        try:
            summary_df = model.get_summary(
                return_period=return_periods,
                return_period_size='365.2425D',
                alpha = 0.95


            )

            # Reset index to make return_period a column
            summary_df = summary_df.reset_index().rename(columns={'index': 'return period'})
            # Add longitude_latitude column
            summary_df['longitude_latitude'] = lon_lat
            
            # Append to list of summaries
            all_summaries.append(summary_df)
            
            # Update results_by_return_period for consistency
            for rp in return_periods:
                try:
                    ret_level_tuple = model.get_return_value(return_period=rp)
                    ret_level = ret_level_tuple[0]
                    results_by_return_period[rp].append({
                        'longitude_latitude': lon_lat,
                        'return_level': ret_level
                    })
                except Exception as e:
                    print(f"Failed for {filename} at {rp} years: {e}")
                    
        except Exception as e:
            print(f"Failed to get summary for {lon_lat}: {e}")
            continue

# Combine and pivot all summaries
if all_summaries:
    combined_df = pd.concat(all_summaries, ignore_index=True)
    
    # Pivot to get return periods as columns
    pivoted_df = combined_df.pivot(
        index='longitude_latitude',
        columns='return period',
        values='return value'
    ).reset_index()
    
    # Rename columns to match desired format
    pivoted_df.columns = [
        'longitude_latitude' if col == 'longitude_latitude'
        else f'return_level_{int(col)}years' for col in pivoted_df.columns
    ]
    
    # Save to a single CSV
    summary_output_path = os.path.join(output_dir, 'all_grid_summaries.csv')
    pivoted_df.to_csv(summary_output_path, index=False)
    
    print(f"Saved combined summary to {summary_output_path}")
else:
    print("No summaries were generated.")


'''

            #print(summary_df)
            
            # Reset index to make return_period a column
            summary_df = summary_df.reset_index().rename(columns={'index': 'return period'})
            # Add longitude_latitude column
            summary_df['longitude_latitude'] = lon_lat
            
            # Pivot the dataframe to have return periods as columns
            pivoted_df = summary_df.pivot(
                index='longitude_latitude',
                columns='return period',
                values='return value'
            ).reset_index()
            
            # Rename columns to match desired format
            pivoted_df.columns = [
                'longitude_latitude' if col == 'longitude_latitude' 
                else f'return_level_{int(col)}years' for col in pivoted_df.columns
            ]
            
            # Save summary to CSV
            summary_output_path = os.path.join(output_dir, f'summary_{lon_lat}.csv')
            pivoted_df.to_csv(summary_output_path, index=False)
        
        except Exception as e:
            print(f"Failed to get summary for {lon_lat}: {e}")'''

'''
        # Initialize result dictionary for this grid point
        results[lon_lat] = {'longitude_latitude': lon_lat}

        # Estimate return levels
        for rp in return_periods:
            try:
                ret_level_tuple = model.get_return_value(return_period=rp)
                ret_level = ret_level_tuple[0]
                results[lon_lat][f'return_level_{rp}years'] = ret_level
            except Exception as e:
                print(f"Failed for {filename} at {rp} years: {e}")
                results[lon_lat][f'return_level_{rp}years'] = None



# Convert results to DataFrame
result_df = pd.DataFrame(list(results.values()))

# Save combined results to a single CSV
output_path = os.path.join(output_dir, 'return_levels_all_periods.csv')
result_df.to_csv(output_path, index=False)
'''

'''
        # Estimate return levels
        for rp in return_periods:
            try:
                ret_level_tuple = model.get_return_value(return_period=rp)
                ret_level = ret_level_tuple[0]
                results_by_return_period[rp].append({
                    'longitude_latitude': lon_lat,
                    'return_level': ret_level
                })
            except Exception as e:
                print(f"Failed for {filename} at {rp} years: {e}")

# Save results for each return period
for rp, results in results_by_return_period.items():
    result_df = pd.DataFrame(results)
    output_path = os.path.join(output_dir, f'return_level_{rp}years.csv')
    result_df.to_csv(output_path, index=False)'''

Saved combined summary to /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/all_grid_summaries.csv


'\n        # Estimate return levels\n        for rp in return_periods:\n            try:\n                ret_level_tuple = model.get_return_value(return_period=rp)\n                ret_level = ret_level_tuple[0]\n                results_by_return_period[rp].append({\n                    \'longitude_latitude\': lon_lat,\n                    \'return_level\': ret_level\n                })\n            except Exception as e:\n                print(f"Failed for {filename} at {rp} years: {e}")\n\n# Save results for each return period\nfor rp, results in results_by_return_period.items():\n    result_df = pd.DataFrame(results)\n    output_path = os.path.join(output_dir, f\'return_level_{rp}years.csv\')\n    result_df.to_csv(output_path, index=False)'

In [10]:
!pip install cartopy

In [11]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from rasterio.mask import mask
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature


In [14]:
def csv_to_tiff_png(csv_path, shapefile_path, output_dir, resolution=0.18, return_periods=[25, 50, 100, 150, 250]):
    # Read shapefile
    try:
        boundary = gpd.read_file(shapefile_path)
        if boundary.crs != 'EPSG:4326':
            boundary = boundary.to_crs(epsg=4326)
    except Exception as e:
        raise ValueError(f"Error reading shapefile: {e}")

    # Check shapefile bounds
    bounds = boundary.total_bounds  # [minx, miny, maxx, maxy]
    lon_min, lat_min, lon_max, lat_max = bounds
    print(f"Shapefile bounds for {csv_path}: {bounds}")
    if not all(np.isfinite([lon_min, lat_min, lon_max, lat_max])):
        raise ValueError(f"Invalid shapefile bounds: {bounds}")

    # Read CSV file
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        raise ValueError(f"Error reading CSV file {csv_path}: {e}")

    # Verify required columns
    required_columns = ['longitude_latitude'] + [f'return_level_{rp}years' for rp in return_periods]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"Warning: Missing columns in {csv_path}: {missing_columns}. Skipping these return periods.")
        return_periods = [rp for rp in return_periods if f'return_level_{rp}years' in df.columns]

    if not return_periods:
        raise ValueError(f"No valid return period columns found in {csv_path}")

    # Split longitude_latitude into lon and lat
    def split_coords(coord):
        if pd.isna(coord) or not isinstance(coord, str):
            return None, None
        parts = coord.replace('grid_', '').split('_')
        if len(parts) != 2:
            return None, None
        try:
            return float(parts[0]), float(parts[1])
        except ValueError:
            return None, None

    # Apply splitting and filter invalid rows
    coords = df['longitude_latitude'].apply(split_coords)
    df['lon'] = [c[0] for c in coords]
    df['lat'] = [c[1] for c in coords]

    # Print sample of invalid rows
    invalid_rows = df['lon'].isna() | df['lat'].isna()
    if invalid_rows.any():
        print(f"Warning: {invalid_rows.sum()} rows in {csv_path} with invalid longitude_latitude format were skipped")
        print(f"Sample of invalid longitude_latitude values in {csv_path}:")
        print(df[invalid_rows]['longitude_latitude'].head(10))
    df = df[~invalid_rows]

    if df.empty:
        raise ValueError(f"No valid coordinate data after filtering in {csv_path}")

    # Check coordinate ranges
    print(f"Longitude range in {csv_path}: min={df['lon'].min()}, max={df['lon'].max()}")
    print(f"Latitude range in {csv_path}: min={df['lat'].min()}, max={df['lat'].max()}")

    # Check if coordinates are within shapefile bounds
    if not (df['lon'].between(lon_min, lon_max).any() and df['lat'].between(lat_min, lat_max).any()):
        print(f"Warning: No coordinates in {csv_path} fall within shapefile bounds {bounds}")

    # Process each return period
    for rp in return_periods:
        return_level_col = f'return_level_{rp}years'
        print(f"Processing return period {rp} years...")

        # Check return_level data
        if not np.all(np.isfinite(df[return_level_col])):
            print(f"Warning: Non-finite values in {return_level_col} for {csv_path}")
            temp_df = df[np.isfinite(df[return_level_col])]
        else:
            temp_df = df

        if temp_df.empty:
            print(f"No valid data for {return_level_col} in {csv_path}. Skipping.")
            continue

        # Calculate grid dimensions
        cols = int((lon_max - lon_min) / resolution) + 1
        rows = int((lat_max - lat_min) / resolution) + 1

        # Create empty grid
        grid = np.full((rows, cols), np.nan)

        # Populate grid with return_level values
        for _, row in temp_df.iterrows():
            col = int((row['lon'] - lon_min) / resolution)
            row_idx = int((lat_max - row['lat']) / resolution)
            if 0 <= row_idx < rows and 0 <= col < cols:
                grid[row_idx, col] = row[return_level_col]

        # Define transform for GeoTIFF
        transform = from_origin(lon_min, lat_max, resolution, resolution)

        # Write to temporary TIFF
        temp_tiff = f'temp_output_{rp}years.tiff'
        try:
            with rasterio.open(
                temp_tiff,
                'w',
                driver='GTiff',
                height=rows,
                width=cols,
                count=1,
                dtype=grid.dtype,
                crs='EPSG:4326',
                transform=transform,
                nodata=np.nan,
            ) as dst:
                dst.write(grid, 1)
        except Exception as e:
            print(f"Error writing temporary TIFF for {return_level_col}: {e}")
            continue

        # Mask the TIFF to the Philippines boundary
        try:
            with rasterio.open(temp_tiff) as src:
                masked_data, masked_transform = mask(src, boundary.geometry, crop=True, nodata=np.nan)
            width, height = masked_data.shape[2], masked_data.shape[1]
            west, south, east, north = (
                masked_transform.c,
                masked_transform.f + masked_transform.e * height,
                masked_transform.c + masked_transform.a * width,
                masked_transform.f,
            )
            if not all(np.isfinite([west, east, south, north])):
                raise ValueError(f"Invalid masked extent for {return_level_col}: [{west}, {east}, {south}, {north}]")
        except Exception as e:
            print(f"Error masking TIFF for {return_level_col}: {e}")
            continue

        # Write final TIFF
        tiff_output = os.path.join(output_dir, f'return_level_{rp}years.tiff')
        try:
            with rasterio.open(
                tiff_output,
                'w',
                driver='GTiff',
                height=masked_data.shape[1],
                width=masked_data.shape[2],
                count=1,
                dtype=masked_data.dtype,
                crs='EPSG:4326',
                transform=masked_transform,
                nodata=np.nan,
            ) as dst:
                dst.write(masked_data[0], 1)
        except Exception as e:
            print(f"Error writing final TIFF for {return_level_col}: {e}")
            continue

        # Check if masked_data contains valid values
        if np.all(np.isnan(masked_data[0])):
            print(f"Warning: No valid data in masked raster for {return_level_col}. Skipping PNG generation.")
            continue

        # Define custom colormap and normalization for wind gust
        colors = [
            "#FFFFFF",  
            "#00CED1",  
            "#3a5f3a",  
            "#73be73",  
            "#90ee90",  
            "#FF4500",  
            "#F0A040",  
            "#D07040",  
            "#A05050",  
            "#803080",  
            "#A040A0",  
            "#C080C0",  
            "#D0A0D0", 
            "#E0C0E0",  
            "#F0E0F0"   
        ]
        bounds = [0, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 30, 35]
        cmap = ListedColormap(colors)
        norm = BoundaryNorm(bounds, cmap.N)

        # Create PNG visualization with cartopy
        fig = plt.figure(figsize=(12, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([115, 130, 5, 20], crs=ccrs.PlateCarree())  # Philippines region
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_facecolor('white')

        # Plot return_level data
        cmap.set_bad('white')
        im = ax.imshow(
            masked_data[0],
            cmap=cmap,
            norm=norm,
            transform=ccrs.PlateCarree(),
            extent=[west, east, south, north],
            origin='upper',
        )
        plt.colorbar(im, ax=ax, label='Wind Gust (m/s)', orientation='vertical', ticks=bounds[:-1])

        # Overlay Philippines boundary
        boundary.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)

        # Add gridlines
        ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)

        # Set title
        ax.set_title(f'Wind Gust Return Level Map (Philippines, {rp} years)')

        # Save PNG
        png_output = os.path.join(output_dir, f'return_level_{rp}years.png')
        try:
            plt.savefig(png_output, dpi=300, bbox_inches='tight')
        except Exception as e:
            print(f"Error saving PNG for {return_level_col}: {e}")
        plt.close()

        print(f"Generated {tiff_output} and {png_output}")

if __name__ == "__main__":
    # Input parameters
    csv_path = '/Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/all_grid_summaries.csv'
    shapefile = 'PHL_adm0.shp'
    output_dir = '/Users/jackywong1105/Desktop/Project 1/WindReturnMap/results'

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Process the CSV
    print(f"Processing {csv_path}...")
    csv_to_tiff_png(csv_path, shapefile, output_dir)
    print("Processing complete.")

Processing /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/all_grid_summaries.csv...
Shapefile bounds for /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/all_grid_summaries.csv: [116.94999   5.04917 126.59804  19.39111]
Longitude range in /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/all_grid_summaries.csv: min=116.69999999999979, max=126.59999999999923
Latitude range in /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/all_grid_summaries.csv: min=5.099999999999948, max=19.399999999999956
Processing return period 25 years...
Generated /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/return_level_25years.tiff and /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/return_level_25years.png
Processing return period 50 years...
Generated /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/return_level_50years.tiff and /Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/return_level_50years.png
Proces